In [4]:
import numpy as np
from functools import reduce

#全连接层实现类
class FullConnectedLayer(object):
    def __init__(self,input_size,output_size,activator):
        """input_size本层输入向量的维度，out_put_size本层输出向量的维度"，activator激活函数"""
        self.input_size = input_size
        self.output_size = output_size
        self.activator = activator
        #权重数组W
        self.W = np.random.uniform(-0.1,0.1,(output_size,input_size))
        #偏置项b
        self.b = np.zeros((output_size,1))
        #输出向量
        self.output = np.zeros((output_size,1))

    def forward(self,input_array):
        """前向计算。imput_array输入向量，维度必须等于input_size"""
        self.input = input_array
        self.output = self.activator.forward(np.dot(self.W,input_array)+self.b) # f(forward()的返回值)

    def backward(self,delta_array):
        """反向计算W和b的梯度。delta_array：从上一层（靠近输出层方向）传递过来的误差项"""
        self.delta = self.activator.backward(self.input)*np.dot(self.W.T,delta_array) # 损失函数对本层净输入（即 z = W·x + b）的偏导数
        self.W_grad = np.dot(delta_array,self.input.T)
        self.b_grad = delta_array

    def update(self,learning_rate):
        """使用梯度下降算法更新权重"""
        self.W += learning_rate*self.W_grad
        self.b += learning_rate*self.b_grad

    def dump(self):
        print('W:%s\n b:%s'%(self.W,self.b))

In [5]:
# Sigmoid激活函数类
class SigmoidActivator(object):
    def forward(self, weighted_input):
        return 1.0 / (1.0 + np.exp(-weighted_input))
    def backward(self, output):
        return output * (1 - output)

# 神经网络类
class Network(object):
    def __init__(self, layers):
        """构造函数"""
        self.layers = [] #经过下面的循环后，该列表中存放的是FullConnectedLayer对象
        for i in range(len(layers) - 1):
            self.layers.append(FullConnectedLayer(layers[i], layers[i+1],SigmoidActivator())) #layers[i]第i层神经元个数，layers[i+1]第i+1层神经元个数

    def predict(self,sample):
        """使用神经网络实现预测，sample输入样本"""
        output = sample
        for layer in self.layers:
            layer.forward(output)
            output = layer.output
        return output

    def train(self,labels,data_set,rate,epoch):
        """训练函数"""
        for i in range(epoch):
            for d in range(len(data_set)):
                self.train_one_sample(labels[d],data_set[d],rate)
    def train_one_sample(self,label,sample,rate):
        self.predict(sample)
        self.calc_gradient(label)
        self.update_weight(rate)
            
    def calc_gradient(self,label):
        delta = self.layers[-1].activator.backward(self.layers[-1].output)*(label - self.layers[-1].output)
        for layer in self.layers[::-1]:
            layer.backward(delta)
            delta = layer.delta
        return delta
    def update_weight(self,rate):
        for layer in self.layers:
            layer.update(rate)
    def dump(self):
        for layer in self.layers:
            layer.dump()  
    def loss(self,label,output):
        return 0.5*((label-output)*(label-output)).sum()
            
    def gradient_check(self, sample_feature, sample_label):
        '''梯度检查。network: 神经网络对象，sample_feature: 样本的特征， sample_label: 样本的标签'''
        # 获取网络在当前样本下每个连接的梯度
        self.predict(sample_feature)
        self.calc_gradient(sample_label)
        # 检查梯度
        epsilon = 10e-4
        for fc in self.layers:
            for i in range(fc.W.shape[0]):
                for j in range(fc.W.shape[1]):
                    fc.W[i,j] += epsilon
                    output = self.predict(sample_feature)
                    err1 = self.loss(sample_label, output)
                    fc.W[i,j] -= 2*epsilon
                    output = self.predict(sample_feature)
                    err2 = self.loss(sample_label, output)
                    expect_grad = (err1 - err2) / (2 * epsilon)
                    fc.W[i,j] += epsilon
                    print('weights(%d,%d): expected - actural %.4e - %.4e' % (i, j, expect_grad, fc.W_grad[i,j]))

In [10]:
def transpose(args):
    """将输入数据转换为列向量的形式。args: 一个元组或列表，包含两个元素：(labels, data_set)，每个元素是一个列表。
    返回值：转换后的 (labels, data_set)，其中每个样本被转为形状 (len(),1) 的 NumPy 列向量。
    """
    result = []
    for arg in args:
        inner = []
        for line in arg:
            inner.append(np.array(line).reshape(len(line), 1))
        result.append(inner)
    return result

# 将一个 0~255 之间的整数（或更小的整数，因为只有 8 位掩码）转换为一个长度为 8 的向量
# 编码时用 0.9 表示该位为 1，用 0.1 表示该位为 0；解码时根据向量分量是否大于 0.5 还原出二进制位，再累加得到原始数字
# 常用于神经网络的输出层（或输入层），当类别数量不多（如 8 个以下）时，可以用这种二进制表示替代 one‑hot 编码，从而降低输出维度（8 维可以表示 256 个类别，而 one‑hot 需要 256 维）
class Normalizer(object):
    def __init__(self):
        self.mask = [0x1, 0x2, 0x4, 0x8, 0x10, 0x20, 0x40, 0x80] #用16进制表示的：1,2,4,8,16,32,64,128,256

    def norm(self, number): #将输入数字按位拆分为8个值，每个值对应0.9（位为1）或0.1（位为0）
        data = [0.9 if number & m else 0.1 for m in self.mask]
        return np.array(data).reshape(8, 1)   # 返回 (8,1) 的 NumPy数组

    def denorm(self, vec):
        """将1个8维向量（每个元素是接近0.1或0.9的浮点数）还原为原始整数。"""
        binary = [1 if i > 0.5 else 0 for i in vec] #通过阈值0.5二值化，然后乘以对应权重，最后用reduce求和
        for i in range(len(self.mask)):
            binary[i] = binary[i] * self.mask[i]
        return reduce(lambda x,y: x + y, binary)

def train_data_set():
    """生成自编码器的训练数据：0~255 的整数，每个整数同时作为输入和标签。返回 (labels, data_set)，其中每个元素都是形状 (8,1) 的列向量列表。"""
    normalizer = Normalizer() 
    data_set = []
    labels = []
    for i in range(0, 256):        # 遍历 0 到 255
        n = normalizer.norm(i)      # 将整数 i 编码为 (8,1) 列向量
        data_set.append(n)
        labels.append(n)
    return labels, data_set

def correct_ratio(network):
    """计算训练好的网络在 0~255 所有整数上的还原正确率。network: 已经训练好的神经网络实例（具有 predict 方法）。"""
    normalizer = Normalizer()       # 创建归一化器
    correct = 0.0                   # 正确计数
    for i in range(256):            # 遍历 0~255
        # 将 i 编码 -> 网络预测 -> 解码，如果等于原始 i 则正确
        if normalizer.denorm(network.predict(normalizer.norm(i))) == i:
            correct += 1.0
    print('correct_ratio: %.2f%%' % (correct / 256 * 100)) # 打印正确率百分比

def test():
    """构建8-3-8网络，训练 10 个epoch"""
    labels, data_set = transpose(train_data_set())
    net = Network([8, 3, 8])
    rate = 0.5
    mini_batch = 20
    epoch = 10
    for i in range(epoch):
        net.train(labels, data_set, rate, mini_batch)
        print('after epoch %d loss: %f' % ((i + 1),net.loss(labels[-1], net.predict(data_set[-1]))))
        rate /= 2
    correct_ratio(net)

def gradient_check():
    '''
    梯度检查
    '''
    labels, data_set = transpose(train_data_set())
    net = Network([8, 3, 8])
    net.gradient_check(data_set[0], labels[0])
    return net

In [11]:
test()

after epoch 1 loss: 0.163733
after epoch 2 loss: 0.159869
after epoch 3 loss: 0.202643
after epoch 4 loss: 0.258254
after epoch 5 loss: 0.306930
after epoch 6 loss: 0.338945
after epoch 7 loss: 0.356705
after epoch 8 loss: 0.365277
after epoch 9 loss: 0.369217
after epoch 10 loss: 0.371087
correct_ratio: 5.47%
